In [1]:
#1.Spark Session initialized
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StreamingAssignment") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Session initialized. Version: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 16:56:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session initialized. Version: 3.5.1


In [2]:
#2.JSON Schema contract defined
#3.ReadStream 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

json_schema = StructType([
    StructField("event_timestamp", TimestampType(), True),
    StructField("country", StringType(), True),
    StructField("temperature", DoubleType(), True)
])

input_directory = "/workspace/input_data"
temperature_stream_df = spark.readStream \
    .format("json") \
    .option("header", "true") \
    .schema(json_schema) \
    .load(input_directory)

print(f"Is Streaming DataFrame active? {temperature_stream_df.isStreaming}")

Is Streaming DataFrame active? True


In [3]:
#4.Handling late data and calculating the avg
from pyspark.sql.functions import window, avg

watermarked_df = temperature_stream_df \
    .withWatermark("event_timestamp", "10 minutes")

transformed_df = watermarked_df \
    .groupBy(
        window("event_timestamp", "15 minutes"),
        "country"
    ) \
    .agg(avg("temperature").alias("average_temperature"))

print("15-min Windowing, and 10-min Watermark defined.")

15-min Windowing, and 10-min Watermark defined.


In [4]:
streaming_query = transformed_df.writeStream \
    .format("console") \
    .outputMode("update") \
    .option("truncate", "false") \
    .start()

print("The stream has started")

try:
    streaming_query.awaitTermination()
except KeyboardInterrupt:
    print("\nStream stopped by user.")
    streaming_query.stop()

26/06/15 16:57:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-fd76219b-06c3-4525-a559-b12bd27ccca7. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/15 16:57:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


The stream has started


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+---------+-------------------+
|window                                    |country  |average_temperature|
+------------------------------------------+---------+-------------------+
|{2024-01-15 13:30:00, 2024-01-15 13:45:00}|USA      |14.5               |
|{2024-01-15 10:45:00, 2024-01-15 11:00:00}|USA      |12.8               |
|{2024-01-15 14:15:00, 2024-01-15 14:30:00}|Canada   |-3.9               |
|{2024-01-15 11:00:00, 2024-01-15 11:15:00}|Japan    |9.4                |
|{2024-01-15 11:00:00, 2024-01-15 11:15:00}|USA      |13.3               |
|{2024-01-15 12:15:00, 2024-01-15 12:30:00}|France   |8.5                |
|{2024-01-15 11:45:00, 2024-01-15 12:00:00}|USA      |13.5               |
|{2024-01-15 10:00:00, 2024-01-15 10:15:00}|Japan    |8.7                |
|{2024-01-15 10:15:00, 2024-01-15 10:30:00}|Germany  |3.4                |
|{2

-------------------------------------------
Batch: 1
-------------------------------------------
+------+-------+-------------------+
|window|country|average_temperature|
+------+-------+-------------------+
+------+-------+-------------------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt



Stream stopped by user.


In [ ]:
import os
output_file_path = "/workspace/output_data/parquet_results"
checkpoint_path = "/workspace/output_data/checkpoints"

os.makedirs(output_file_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

def write_to_multiple_sinks(batch_df, batch_id):
    print(f"Processing Batch ID: {batch_id} ")
    
    batch_df.cache()
    
    try:
        print(f"Saving Batch {batch_id} to Filesystem")
        batch_df.write \
            .format("parquet") \
            .mode("append") \
            .save(output_file_path)
        
        print(f"Saving Batch {batch_id} to Hive Table")
        batch_df.write \
            .format("parquet") \
            .mode("append") \
            .saveAsTable("default.country_avg_temperatures")
            
        print(f"Batch {batch_id} processed successfully")
        
    except Exception as e:
        print(f"Error writing Batch {batch_id}: {str(e)}")
        
    finally:
        batch_df.unpersist()

final_query = transformed_df.writeStream \
    .outputMode("update") \
    .foreachBatch(write_to_multiple_sinks) \
    .option("checkpointLocation", checkpoint_path) \
    .start()

try:
    final_query.awaitTermination()
except KeyboardInterrupt:
    print("\nStreaming query stopped.")
    final_query.stop()

26/06/15 16:58:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Processing Batch ID: 0 
Saving Batch 0 to Filesystem


26/06/15 16:58:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Saving Batch 0 to Hive Table


Batch 0 processed successfully
Processing Batch ID: 1 
Saving Batch 1 to Filesystem


Saving Batch 1 to Hive Table


Batch 1 processed successfully
